In [ ]:
#| hide
from speaking_plants.clean import *
from speaking_plants.leaf import *

# speaking-plants

> Detecting water stress in coffee plants with IoT sensors: thermal cameras, soil moisture and relative humidity

This project tests whether low-cost sensors can detect water stress in coffee plants before it becomes visible. A plant short of water closes its stomata, transpires less, and its leaves warm above the air. A thermal camera can see that warming.

Four sensor nodes, `SP_01` to `SP_04`, each watch one plant in a greenhouse. Every 5 minutes, each node records a burst of about 5 thermal images from a 32 by 24 pixel MLX90640 camera. It also records the moisture and temperature of the plant's soil. `SP_01` also carries a Ruuvi sensor for air temperature, relative humidity and pressure, which applies to all plants. Each node writes its readings to a CSV log on an SD card.

## Contents

| Notebook | Module | What it does |
|---|---|---|
| [Clean](https://franckalbinet.github.io/speaking-plants/clean.html) | `speaking_plants.clean` | Reads the SD-card logs into one table, flags faulty values, combines each burst into one measurement, and shares the Ruuvi readings with every node. |
| [Leaf temperature](https://franckalbinet.github.io/speaking-plants/leaf.html) | `speaking_plants.leaf` | Builds daily leaf temperature series per node, with or without a leaf mask, smoothed, as a slope, and as leaf minus air temperature. Splits time into day and night from sunrise and sunset. Shows where camera scenes change, and compares nodes with biases, fitted lines and scatter plots. |
| [Logbook](https://franckalbinet.github.io/speaking-plants/logbook.html) | `speaking_plants.logbook` | Reads the deployment and event tables in `logbook/`, links logged rows to deployments and plants, and lists where the logbook and the logs disagree. |
| [Environment](https://franckalbinet.github.io/speaking-plants/environment.html) | none | Plots air humidity and soil moisture over a batch, and compares soil drying with leaf temperature. |

## Install

Install the latest version from GitHub:

```sh
pip install git+https://github.com/franckalbinet/speaking-plants.git
```

## How to use

The CSV logs are not in the repository. Put the logs of each batch in its own folder under `data/` at the project root, such as `data/b01/`. This example runs from `nbs/`. It reads batch 1 and returns one leaf temperature series per node and full day, without a leaf mask:

In [ ]:
#| eval: false
from pathlib import Path
from speaking_plants.clean import *
from speaking_plants.leaf import *

logs = sorted(Path('../data/b01').glob('dados_*.csv'))
mdf = share_ruuvi(measurements(clean(read_logs(logs))))
days = full_days(mdf)
ls = leaf_series(mdf, no_masks(days))
ls[['node_id', 'time', 't_leaf', 't_smooth', 't_air', 'dt_leaf_air', 'slope']].head()

,node_id,time,t_leaf,t_smooth,t_air,dt_leaf_air,slope
0,SP_01,2026-09-25 00:00:00,22.045,22.023571,NaN,NaN,-0.010531
1,SP_01,2026-09-25 00:05:00,21.955,21.999945,19.94,2.015,-0.006806
2,SP_01,2026-09-25 00:10:00,21.980,21.973666,19.93,2.050,-0.004375
3,SP_01,2026-09-25 00:15:00,21.960,21.945749,19.91,2.050,-0.003236
4,SP_01,2026-09-25 00:20:00,21.935,21.917208,NaN,NaN,-0.003391


`mdf` has one row per node and measurement. `ls` has one row per node and 5-minute step of each full day.

## Develop

The notebooks in `nbs/` are the source. `nbdev` exports them to the `speaking_plants` package and builds the documentation site. With [uv](https://docs.astral.sh/uv/):

```sh
uv sync                # install the package in editable mode, with its dependencies
uv run nbdev-export    # export the notebooks to speaking_plants/
uv run nbdev-test      # run the notebooks as tests
uv run nbdev-clean     # strip notebook metadata before committing
```

Cells that read the logs have `#| eval: false`, so the tests run without the data.